<a href="https://colab.research.google.com/github/Meghan1432md/ML-lab/blob/main/ML_ORANGE_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PROBLEM 62: NCA Distance Learner - COMPLETE SOLUTION
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD WINE DATASET
print("=== STEP 1: Loading Wine Dataset ===")
wine = fetch_ucirepo(id=109)  # Wine recognition dataset
X = wine.data.features.values
y = wine.data.targets.values.ravel()
feature_names = wine.data.features.columns

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")
print("Features:", list(feature_names[:5]), "...")

# 2. SPLIT DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. BASELINE: kNN on ORIGINAL FEATURES (3D plot first 3 features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_original = KNeighborsClassifier(n_neighbors=3)
knn_original.fit(X_train_scaled, y_train)
y_pred_original = knn_original.predict(X_test_scaled)
acc_original = accuracy_score(y_test, y_pred_original)

print(f"\n=== STEP 2: kNN Baseline Accuracy ===")
print(f"Original features kNN accuracy: {acc_original:.3f} ({acc_original*100:.1f}%)")

# 4. NCA TRANSFORMATION
print("\n=== STEP 3: Applying NCA Transformation ===")
nca = NeighborhoodComponentsAnalysis(n_components=3, random_state=42)  # Reduce to 3D
X_train_nca = nca.fit_transform(X_train_scaled, y_train)
X_test_nca = nca.transform(X_test_scaled)

# 5. kNN on NCA FEATURES
knn_nca = KNeighborsClassifier(n_neighbors=3)
knn_nca.fit(X_train_nca, y_train)
y_pred_nca = knn_nca.predict(X_test_nca)
acc_nca = accuracy_score(y_test, y_pred_nca)

print(f"NCA features kNN accuracy: {acc_nca:.3f} ({acc_nca*100:.1f}%)")
print(f"IMPROVEMENT: {((acc_nca-acc_original)/acc_original)*100:.1f}% ↑")

# 6. VISUALIZATIONS
fig = plt.figure(figsize=(15, 5))

# Plot 1: 3D Before NCA (first 3 original features)
ax1 = fig.add_subplot(131, projection='3d')
scatter1 = ax1.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], X_train_scaled[:, 2],
                       c=y_train, cmap='viridis', s=50)
ax1.set_title('Before NCA\n(Original Features)')
ax1.set_xlabel('Feature 0'); ax1.set_ylabel('Feature 1'); ax1.set_zlabel('Feature 2')
plt.colorbar(scatter1)

# Plot 2: 3D After NCA
ax2 = fig.add_subplot(132, projection='3d')
scatter2 = ax2.scatter(X_train_nca[:, 0], X_train_nca[:, 1], X_train_nca[:, 2],
                       c=y_train, cmap='viridis', s=50)
ax2.set_title('After NCA\n(Transformed Features)')
ax2.set_xlabel('NCA 0'); ax2.set_ylabel('NCA 1'); ax2.set_zlabel('NCA 2')
plt.colorbar(scatter2)

# Plot 3: kNN Accuracy Comparison
ax3 = fig.add_subplot(133)
accuracies = [acc_original, acc_nca]
methods = ['Original\nFeatures', 'NCA\nFeatures']
bars = ax3.bar(methods, accuracies, color=['skyblue', 'lightgreen'], alpha=0.8)
ax3.set_ylabel('kNN Accuracy')
ax3.set_title('kNN Accuracy: Before vs After NCA')
ax3.set_ylim(0, 1)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('nca_wine_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== ALL PLOTS SAVED: nca_wine_results.png ===")
print("\n✅ ASSIGNMENT COMPLETE!")


ModuleNotFoundError: No module named 'ucimlrepo'